# ?? ?? ????

- ??: ?? ??? ??? ?? ?? ??? ?????.
- ??: ?? ?? ??? ?? ??
- ??: ?? ??? ?? ??
- ?? ??: ?? ?????? ?? ???? ?????.


In [ ]:
import pandas as pd

# 데이터 불러오기

In [ ]:
df = pd.read_excel('./data/final_all_data.xlsx')
df

In [ ]:
df.columns

In [ ]:
df['유투버'].unique()

In [ ]:
dates = ['22_08', '22_09', '22_10', '22_11', '22_12', '23_01', '23_02', '23_03', '23_04', '23_05', '23_06', '23_07', '23_08']

In [ ]:
vars = ['구독자점수', '조회수점수', '충성도점수', '구독자 성장율점수', '감성점수점수', '평균_영상간격점수',
       '평균 영상 좋아요 수 점수', '영상개수 점수', '등급' ]

In [ ]:
cols = []
for date in dates:
    for var in vars:
        cols.append(date + '_' + var)
cols

# 데이터프레임 형태 변환

In [ ]:
df.columns

In [ ]:
list(df.columns).index('구독자점수')

In [ ]:
df_new = pd.DataFrame(columns=['유투버'] + cols)
for name_idx, name in enumerate(df['유투버'].unique()):
    df_new.loc[name_idx, '유투버'] = name
    for i, date in enumerate(dates):
        start_name = df_new.columns[i*9+1]
        end_name = df_new.columns[(i+1)*9]
        df_new.loc[name_idx, start_name:end_name] = df[df['유투버'] == name].iloc[i, [15, 16, 17, 18, 19, 20, 21, 22, -1]].values

In [ ]:
df_new

In [ ]:
df_new['유투버'].unique()

In [ ]:
df_new.columns

In [ ]:
df_new.isnull().sum().sum()

# 1개월치 데이터로 예측

## 등급 자체를 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = df_new.iloc[:, -18:-10]
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = df_new.iloc[:, -27:-19]
X_train

In [ ]:
def label_encode_dataframe(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 1, 'B+': 2, 'B0': 3,
        'C+': 4, 'C0': 5, 'D+': 6, 'D0': 7,
        'E': 8, 'F': 9
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe(y_test, y_test.columns[0])
label_encode_dataframe(y_train, y_train.columns[0])
y_test

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 간소화된 등급 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = df_new.iloc[:, -18:-10]
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
def label_encode_dataframe_2(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 1, 'B0': 1,
        'C+': 2, 'C0': 2, 'D+': 3, 'D0': 3,
        'E': 4, 'F': 4
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_2(y_test, y_test.columns[0])
label_encode_dataframe_2(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = df_new.iloc[:, -27:-19]
X_train

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 인기, 비인기 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = df_new.iloc[:, -18:-10]
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
def label_encode_dataframe_3(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 0, 'B0': 0,
        'C+': 1, 'C0': 1, 'D+': 1, 'D0': 1,
        'E': 1, 'F': 1
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_3(y_test, y_test.columns[0])
label_encode_dataframe_3(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = df_new.iloc[:, -27:-19]
X_train

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

# 3개월치 데이터로 예측

## 등급 자체를 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
label_encode_dataframe(y_test, y_test.columns[0])
label_encode_dataframe(y_train, y_train.columns[0])
y_test

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 간소화된 등급 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
def label_encode_dataframe_2(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 1, 'B0': 1,
        'C+': 2, 'C0': 2, 'D+': 3, 'D0': 3,
        'E': 4, 'F': 4
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_2(y_test, y_test.columns[0])
label_encode_dataframe_2(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 인기, 비인기 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]

In [ ]:
X_train = pd.concat((df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
label_encode_dataframe_3(y_test, y_test.columns[0])
label_encode_dataframe_3(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

### 예측 확률 계산

In [ ]:
# df_new['예측확률'] = pd.DataFrame(model_xgb.predict_proba(X_test.values))[1]

In [ ]:
# df_new['인기_비인기_여부'] = y_test

In [ ]:
# df_new

In [ ]:
# df_new.to_excel('./data/등급_예측.xlsx', index=False)

# 6개월치 데이터로 예측

## 등급 자체를 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
label_encode_dataframe(y_test, y_test.columns[0])
label_encode_dataframe(y_train, y_train.columns[0])
y_test

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 간소화된 등급 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]
y_train

In [ ]:
X_train = pd.concat((df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
def label_encode_dataframe_2(df, column_name):
    label_mapping = {
        'A+': 0, 'A0': 0, 'B+': 1, 'B0': 1,
        'C+': 2, 'C0': 2, 'D+': 3, 'D0': 3,
        'E': 4, 'F': 4
    }
    df[column_name] = df[column_name].map(label_mapping)
    return df

In [ ]:
label_encode_dataframe_2(y_test, y_test.columns[0])
label_encode_dataframe_2(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg,  rf, model_xgb, model_lgb]:
    clf = model.fit(X_train, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred, average='weighted')
    recall = recall_score(y_test, pred, average='weighted')
    f1 = f1_score(y_test, pred, average='weighted')
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df

## 인기, 비인기 예측

In [ ]:
y_test = df_new.iloc[:, [-1]]
y_test

In [ ]:
X_test = pd.concat((df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19], df_new.iloc[:, -18:-10]), axis=1)
X_test

In [ ]:
y_train = df_new.iloc[:, [-10]]

In [ ]:
X_train = pd.concat((df_new.iloc[:, -72:-64], df_new.iloc[:, -63:-55], df_new.iloc[:, -54:-46], df_new.iloc[:, -45:-37], df_new.iloc[:, -36:-28], df_new.iloc[:, -27:-19]), axis=1)
X_train

In [ ]:
label_encode_dataframe_3(y_test, y_test.columns[0])
label_encode_dataframe_3(y_train, y_train.columns[0])
y_test

In [ ]:
X_train = X_train.astype('float')
X_test = X_test.astype('float')

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn import svm
import xgboost as xgb
import lightgbm as lgb

# 사용할 모델 정의
log_reg = LogisticRegression(random_state=42, n_jobs=-1)
SGD = SGDClassifier(random_state=42, n_jobs=-1)
DTree = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_xgb = xgb.XGBClassifier(random_state=42, n_jobs=-1)
model_lgb = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
models = []
acc_scores = []
recall_scores = []
precision_scores = []  
f1_scores = []  
for model in [log_reg, rf, model_xgb, model_lgb]:
    clf = model.fit(X_train.values, y_train)
    pred = clf.predict(X_test.values)
    # 평가 지표 계산

    acc = accuracy_score(y_test, pred)
    precsion = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    models.append(model.__class__.__name__)
    acc_scores.append(acc)
    precision_scores.append(precsion)
    recall_scores.append(recall)
    f1_scores.append(f1)
result_df = pd.DataFrame({'Model': models, 'Acurracy': acc_scores, 'Recall': recall_scores, 'Precision': precision_scores, 'f1': f1_scores}).reset_index(drop=True)
result_df